In [ ]:
import matplotlib.pyplot as plt
import pandas
import functools 
import datetime

import pmana.utils
import pmana.purity

#### Look at the purity of one measurement

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

YEAR = '2026'
MONTH = 'Mar'
MEASUREMENT = f'08_15_48'
FILE_PATH = f"../../data/coldbox/{YEAR}_{MONTH}/Record_{YEAR}_{MONTH}_{MEASUREMENT}.csv"
# extract a measurement 
Data = pmana.utils.io.ExtractSingleMeasurement(
    FILE_PATH,
    IS_CSV = True,
    COL_NAMES = ['binCenter', 'F1', 'F2', 'F3', 'F4'],
    DELIMITER = ","
)
print(f"There were {len(Data)} used channels in this measurement.")

CHs = [1, 2, 3, 4]
for ch in CHs:
    # plot channel data
    pmana.utils.plotting.PlotSingleChannel(
        Data[ch-1],
        ax,
        channel = ch-1,
        rebin = True,
        debug = False,
        DISPLAY_FIT = False
    )

ax = pmana.utils.plotting.UpdateMatplotlibStyle(
    ax,
    "Pulse height [V]",
    "Counts [#]"
)

ax.legend()
# ax.set_yscale('log')
ax.set_title(f"{MONTH} {YEAR} / {MEASUREMENT.replace('_','-')}")
plt.show()
fig.savefig(f"../../plots/coldbox/RawData_{YEAR}_{MONTH}.png", dpi=300, bbox_inches='tight')

#### Purity analysis

In [ ]:
# configure analysis
ANALYSIS_CONFIGURATION = pmana.purity.config.DEFAULT_ANALYSIS_CONFIGURATION
ANALYSIS_CONFIGURATION['LongICPeakSearchLimits'] = (0.15, 0.5)   # long Bi-PrM has a significant tail...
ANALYSIS_CONFIGURATION['ShortICPeakSearchLimits'] = (0.4, 0.8)  # short Bi-PrM has a test pulse...
ANALYSIS_CONFIGURATION['InnerLongChannel'] = 2
ANALYSIS_CONFIGURATION['OuterLongChannel'] = 3
ANALYSIS_CONFIGURATION['InnerShortChannel'] = 0
ANALYSIS_CONFIGURATION['OuterShortChannel'] = 1

# configure calibration factors
# the important thing is inter-calibrating
# inner and outer anodes within a PrM
# e.g., by aligning their test-pulse
ANALYSIS_CONFIGURATION['InnerLongCalibration'] = 1.
ANALYSIS_CONFIGURATION['OuterLongCalibration'] = 1.
ANALYSIS_CONFIGURATION['InnerShortCalibration'] = 1.
ANALYSIS_CONFIGURATION['OuterShortCalibration'] = 1.

In [ ]:
# analyze campaign
TIME_DIR = f"2026_Mar"
PATH_CAMPAIGN = f"../../data/coldbox/{TIME_DIR}"

### long PrM
Output = pmana.utils.iterators.IterateCERN_CSV(
    PATH_CAMPAIGN,  ###< path to data
    # pmana.utils.anatestdata.AnalyzeMeasurement, 
    functools.partial(
        pmana.purity.ana.ExtractICPeak, 
        PM_TAG = 'Long',
        ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION
    ),  ###< analyzing module, changing some options 
)

# convert top DataFrame and re-format the dataframe
Output = pandas.DataFrame(Output)
Output.columns = [
  "LongIC_Peak", "LongIC_PeakErr", "LongIC_Scale",
  "Date"
]

### short PrM
Output_Short = pmana.utils.iterators.IterateCERN_CSV(
  PATH_CAMPAIGN,  ###< path to data
  # pmana.utils.anatestdata.AnalyzeMeasurement, 
  functools.partial(
      pmana.purity.ana.ExtractICPeak, 
      PM_TAG = 'Short',
      ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION
  ),  ###< analyzing module, changing some options 
)

# convert top DataFrame and re-format the dataframe
Output_Short = pandas.DataFrame(Output_Short)
Output_Short.columns = [
  "ShortIC_Peak", "ShortIC_PeakErr", "ShortIC_Scale",
  "Date"
]

### merge
Output = pandas.merge(Output, Output_Short, on = 'Date', how='inner')

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
Output['Lifetime'], Output['Lifetime_Err'] = pmana.purity.ana.GetLifetime_DoublePrM(
  Output.ShortIC_Peak, Output.LongIC_Peak,
  Output.ShortIC_PeakErr, Output.LongIC_PeakErr,
  SHORT_DRIFT_LENGTH = 40, LONG_DRIFT_LENGTH = 400
)
# Output

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3), nrows=1, layout='tight')

ax.errorbar(
  Output['Date'], Output['Lifetime'], 
  yerr = Output['Lifetime_Err'],
  c='black', label='Double Bi-PrM (4 + 40 cm)',
  ls='', marker='.'
)

# ax.set_yscale('log')
ax.set_ylim(170, 240)

# gfx
pmana.utils.plotting.UpdateMatplotlibStyle(ax, None, 'Lifetime [μs]')
ax.legend(handlelength=1, loc='upper right', fancybox=False)
ax.tick_params(axis='x', labelrotation=30)
ax.grid(which='major', alpha=0.5)
ax.grid(which='minor', ls='--', alpha=0.3)

fig.suptitle(f'CERN-Coldbox data', fontsize=16)

plt.show()
fig.savefig(f"../../plots/coldbox/Lifetime_{TIME_DIR}.png", dpi=300, bbox_inches='tight')

#### Purity analysis, but save spectra

In [ ]:
# configure analysis
ANALYSIS_CONFIGURATION = pmana.purity.config.DEFAULT_ANALYSIS_CONFIGURATION
ANALYSIS_CONFIGURATION['LongICPeakSearchLimits'] = (0.15, 0.5)   # long Bi-PrM has a significant tail...
ANALYSIS_CONFIGURATION['ShortICPeakSearchLimits'] = (0.4, 0.8)  # short Bi-PrM has a test pulse...
ANALYSIS_CONFIGURATION['InnerLongChannel'] = 2
ANALYSIS_CONFIGURATION['OuterLongChannel'] = 3
ANALYSIS_CONFIGURATION['InnerShortChannel'] = 0
ANALYSIS_CONFIGURATION['OuterShortChannel'] = 1

# configure calibration factors
# the important thing is inter-calibrating
# inner and outer anodes within a PrM
# e.g., by aligning their test-pulse
CalibrationFactors = {
  0: 1.,   #
  1: 1.,   #
  2: 1.,   # inner long channel
  3: 1.  # outer long channel
}

In [ ]:
# analyze campaign
TIME_DIR = f"2026_Mar"
PATH_CAMPAIGN = f"../../data/coldbox/{TIME_DIR}"

Output = pmana.utils.iterators.IterateCERN_CSV(
    PATH_CAMPAIGN,  ###< path to data
    # pmana.utils.anatestdata.AnalyzeMeasurement, 
    functools.partial(
        pmana.purity.ana.ExtractICPeak, 
        PM_TAG = 'Long',
        SAVE_SPECTRA = True, ###< save the spectra on the dataframe
        ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION
    )  ###< analyzing module, changing some options 
)

# convert top DataFrame
Output = pandas.DataFrame(Output)

# re-format the dataframe
Output.columns = [
  "LongIC_Peak", "LongIC_PeakErr", "LongIC_Scale",
  "xIC", "IC",
  "Date"
]

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

YEAR = '2026'
MONTH = 'Mar'
MEASUREMENT = f'08_15_48'
FILE_PATH = f"../../data/coldbox/{YEAR}_{MONTH}/Record_{YEAR}_{MONTH}_{MEASUREMENT}.csv"

# extract a measurement 
Data = pmana.utils.io.ExtractSingleMeasurement(
    FILE_PATH,
    IS_CSV = True,
    COL_NAMES = ['binCenter', 'F1', 'F2', 'F3', 'F4'],
    DELIMITER = ","
)
print(f"There were {len(Data)} used channels in this measurement.")

CHs = [
    ANALYSIS_CONFIGURATION['InnerLongChannel'],
    ANALYSIS_CONFIGURATION['OuterLongChannel'] 
]
for ch in CHs:
    # plot channel data
    pmana.utils.plotting.PlotSingleChannel(
        Data[ch-1],
        ax,
        channel = ch-1,
        rebin = False,
        debug = False,
        DISPLAY_FIT = False
    )

ax = pmana.utils.plotting.UpdateMatplotlibStyle(
    ax,
    "Integral [V*t/R] / Pulse height [V]",
    "Counts [#]"
)

# plot the corresponding IC peak
MONTH_ABBR = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
DAY, HOUR, MINUTE = MEASUREMENT.split('_')
EXACT_DATE = datetime.datetime(
    int(YEAR), 
    int(MONTH_ABBR[MONTH]),
    int(DAY),
    int(HOUR),
    int(MINUTE)
)
Mask = Output['Date'] == EXACT_DATE
print("Results for this measurement:\n", Output[Mask])
xIC = Output[Mask].iloc[0]['xIC']
IC = Output[Mask].iloc[0]['IC']
ax.step(
    xIC, 
    IC,
    where = 'mid',
    c = 'red',
    lw = 1,
    label = 'IC'
)

ax.legend(handlelength=1, loc='upper right', frameon=True, fancybox=False)
ax.set_title(f"{MONTH} {YEAR} / {MEASUREMENT}")
plt.show()
fig.savefig(f"../../plots/coldbox/RawData_IC_{YEAR}_{MONTH}.png", dpi=300, bbox_inches='tight')

In [ ]:
Output = pmana.utils.iterators.IterateCERN_CSV(
  PATH_CAMPAIGN,  ###< path to data
  # pmana.utils.anatestdata.AnalyzeMeasurement, 
  functools.partial(
      pmana.purity.ana.ExtractICPeak, 
      PM_TAG = 'Short',
      SAVE_SPECTRA = True, ###< save the spectra on the dataframe
      ANALYSIS_CONFIGURATION = ANALYSIS_CONFIGURATION
  ),  ###< analyzing module, changing some options 
)

# convert top DataFrame
Output = pandas.DataFrame(Output)

# re-format the dataframe
Output.columns = [
  "ShortIC_Peak", "ShortIC_PeakErr", "ShortIC_Scale",
  "xIC", "IC",
  "Date"
]

# sort by date
Output.sort_values(
    by = 'Date', 
    inplace = True,
    ignore_index = True
)

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

# extract a measurement 
Data = pmana.utils.io.ExtractSingleMeasurement(
    FILE_PATH,
    IS_CSV = True,
    COL_NAMES = ['binCenter', 'F1', 'F2', 'F3', 'F4'],
    DELIMITER = ","
)
print(f"There were {len(Data)} used channels in this measurement.")

CHs = [3, 4]
for ch in CHs:
    # plot channel data
    pmana.utils.plotting.PlotSingleChannel(
        Data[ch-1],
        ax,
        channel = ch-1,
        rebin = False,
        debug = False,
        DISPLAY_FIT = False
    )

ax = pmana.utils.plotting.UpdateMatplotlibStyle(
    ax,
    "Pulse height [V]",
    "Counts [#]"
)

# plot the corresponding IC peak
MONTH_ABBR = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
DAY, HOUR, MINUTE = MEASUREMENT.split('_')
EXACT_DATE = datetime.datetime(
    int(YEAR), 
    int(MONTH_ABBR[MONTH]),
    int(DAY),
    int(HOUR),
    int(MINUTE)
)
Mask = Output['Date'] == EXACT_DATE
print("Results for this measurement:\n", Output[Mask])
xIC = Output[Mask].iloc[0]['xIC']
IC = Output[Mask].iloc[0]['IC']
ax.step(
    xIC, 
    IC,
    where = 'mid',
    c = 'red',
    lw = 1,
    label = 'IC'
)

ax.legend(handlelength=1, loc='upper right', frameon=True, fancybox=False)
ax.set_title(f"{MONTH} {YEAR} / {MEASUREMENT}")
plt.show()
fig.savefig(f"../../plots/cern/RawData_IC_Short_{YEAR}_{MONTH}.png", dpi=300, bbox_inches='tight')